# Invariant EKF Covariance Calibration

Notebook for the trimmed covariance-calibration run. The next cell configures CUDA float64, paths, output folders, and imports the Torch replay module.


Configure CUDA float64, project paths, output folders, and imports.


In [1]:
# 01 environment: CUDA-first, float64, bundle paths
import copy
import json
import os
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch

def find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "estimation_calibration_cuda" / "invariant_ekf.py").exists():
            return p
    raise FileNotFoundError("project root not found")

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))
DATA_ROOT = Path(os.environ.get("INEKF_DATA_ROOT", ROOT / "data" / "datasets_v0"))
OUT = ROOT / "runs" / "covariance_calibration"
PLOTS = OUT / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

torch.set_default_dtype(torch.float64)
assert torch.cuda.is_available(), "This notebook is CUDA-first; run on the 5090 Laptop GPU."
device = torch.device("cuda")
dtype = torch.float64
torch.manual_seed(0)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

def assert_cuda_float64(*xs):
    for x in xs:
        if isinstance(x, torch.Tensor):
            assert x.device.type == "cuda", x.device
            assert x.dtype == torch.float64, x.dtype

DD = dict(dtype=dtype, device=device)

from estimation_calibration_cuda.invariant_ekf import (
    replay_inekf_torch, start_filter, run_rows, detach_filter, precompute_contact_changes
)

# fixed categorical colors (Okabe-Ito, CVD-safe), one per covariance group
GROUP_ORDER = ["gyro", "accel", "gyro_bias", "accel_bias", "contact_proc", "kin_meas"]
GROUP_COLOR = dict(zip(GROUP_ORDER,
                       ["#0072B2", "#E69F00", "#009E73", "#D55E00", "#CC79A7", "#56B4E9"]))


NVIDIA GeForce RTX 5090 Laptop GPU | torch 2.12.0+cu130


Validate the Torch replay and chunked-continuation behavior.


In [2]:
# 02 replay checks: synthetic sequence, G1 slice, and chunked continuation.
gs = np.load(OUT / "gate_c_golden_synthetic.npz")
gg = np.load(OUT / "gate_c_golden_g1_slice.npz")

def covs_from(z):
    return {k: torch.as_tensor(z[k], **DD) for k in ["Qg", "Qa", "Qbg", "Qba", "Qc"]}

def tensors_from(z):
    return (torch.as_tensor(z["X0"], **DD), torch.as_tensor(z["theta0"], **DD),
            torch.as_tensor(z["P0"], **DD), torch.as_tensor(z["imu"] if "imu" in z
            else z["imu_shifted"], **DD), torch.as_tensor(z["p_meas"], **DD),
            torch.as_tensor(z["R_kin"], **DD))

for name, z, tol in [("synthetic", gs, 1e-10), ("g1_slice", gg, 1e-9)]:
    X0, th0, P0, imu, p, Rk = tensors_from(z)
    assert_cuda_float64(X0, th0, P0, imu, p, Rk)
    out = replay_inekf_torch(X0, th0, P0, covs_from(z), imu, float(z["dt"]),
                                     p, z["flags"], Rk)
    d = max(float((out["R_WB"] - torch.as_tensor(z["R_np"], **DD)).abs().max()),
            float((out["v_W"] - torch.as_tensor(z["v_np"], **DD)).abs().max()),
            float((out["p_W"] - torch.as_tensor(z["p_np"], **DD)).abs().max()),
            float((out["final_P"] - torch.as_tensor(z["final_P"], **DD)).abs().max()))
    print(f"Replay check [{name}]: max abs diff = {d:.2e}")
    assert d < tol, (name, d)
    if name == "synthetic":
        got = out["final_estimated_contact_positions"]
        want = dict(zip(z["final_contact_ids"].tolist(), z["final_contact_cols"].tolist()))
        assert got == want, (got, want)  # dynamic dims preserved after add/remove

# chunked continuation == monolithic (bitwise), with and without precomputed events
X0, th0, P0, imu, p, Rk = tensors_from(gg)
covs = covs_from(gg)
out_m = replay_inekf_torch(X0, th0, P0, covs, imu, float(gg["dt"]), p,
                                   gg["flags"], Rk)
gg_changes = precompute_contact_changes(gg["flags"])
for use_pre in (False, True):
    filt = start_filter(X0, th0, P0, covs, gg["flags"][0], p[0], Rk)
    vs = [filt.X[0:3, 3][None]]
    a = 1
    while a < p.shape[0]:
        b = min(a + 100, p.shape[0])
        kw = dict(changes_list=gg_changes[a:b]) if use_pre else {}
        oc = run_rows(filt, imu[a - 1:b - 1], float(gg["dt"]), p[a:b], gg["flags"][a:b],
                      gg["flags"][a - 1], Rk, **kw)
        vs.append(oc["v_W"])
        detach_filter(filt)
        a = b
    v_chunked = torch.cat(vs)
    assert torch.equal(v_chunked, out_m["v_W"]), f"chunked (pre={use_pre}) must be exact"
print("Gate C: PASS (parity, dynamic dims, chunked continuation exact, "
      "precomputed contact events exact)")


Replay check [synthetic]: max abs diff = 2.25e-14


Replay check [g1_slice]: max abs diff = 3.12e-14


Replay checks: PASS (dynamic dims, chunked continuation, precomputed contact events)


Define the 3x3 SPD covariance modules used for each process and measurement covariance group.


In [3]:
# 03 full-SPD Cholesky covariance modules
class SPD3(torch.nn.Module):
    """3x3 SPD covariance via softplus-diagonal Cholesky factor.

    Off-diagonal Cholesky entries are stored in units of the group's initial
    standard deviation so Adam steps have comparable scale across covariance
    groups while preserving the same SPD form and initialization.
    """

    def __init__(self, init_std, *, floor=1e-12, dtype=torch.float64, device="cuda"):
        super().__init__()
        init_std = torch.as_tensor(init_std, dtype=dtype, device=device).reshape(3)
        raw = torch.zeros(3, 3, dtype=dtype, device=device)
        target_diag = torch.clamp(init_std, min=floor**0.5)
        raw_diag = torch.log(torch.expm1(target_diag))
        raw[0, 0], raw[1, 1], raw[2, 2] = raw_diag
        self.raw_tril = torch.nn.Parameter(raw)
        self.floor = float(floor)
        self.offdiag_scale = float(init_std.mean())

    def L(self):
        tril = torch.tril(self.raw_tril)
        diag = torch.nn.functional.softplus(torch.diagonal(tril)) + self.floor**0.5
        off = (tril - torch.diag(torch.diagonal(tril))) * self.offdiag_scale
        return off + torch.diag(diag)

    def cov(self):
        L = self.L()
        return L @ L.T + self.floor * torch.eye(3, dtype=L.dtype, device=L.device)

    def log_eigs(self):
        return torch.log(torch.linalg.eigvalsh(self.cov()).clamp_min(self.floor))

# initial stds and floors for each covariance group
INIT_STD = {
    "gyro": 0.01, "accel": 0.3, "gyro_bias": 1e-5, "accel_bias": 1e-4,
    "contact_proc": 0.1, "kin_meas": 0.02,
}
FLOOR = {
    "gyro": 1e-10, "accel": 1e-8, "gyro_bias": 1e-16, "accel_bias": 1e-14,
    "contact_proc": 1e-8, "kin_meas": 1e-10,
}
COV_KEY = {"gyro": "Qg", "accel": "Qa", "gyro_bias": "Qbg",
           "accel_bias": "Qba", "contact_proc": "Qc"}

def make_cov_modules():
    return torch.nn.ModuleDict({
        name: SPD3([INIT_STD[name]] * 3, floor=FLOOR[name], device=device)
        for name in GROUP_ORDER
    })

def build_covs(modules):
    covs = {COV_KEY[n]: modules[n].cov() for n in COV_KEY}
    R_kin = modules["kin_meas"].cov()
    return covs, R_kin

# unit checks
_m = make_cov_modules()
for name in GROUP_ORDER:
    C = _m[name].cov()
    assert_cuda_float64(C)
    # softplus floor offset shifts the diagonal by ~2*std*sqrt(floor): allow 1%
    assert float((C - INIT_STD[name] ** 2 * torch.eye(3, **DD)).abs().max()) < 1e-2 * INIT_STD[name] ** 2
    eig = torch.linalg.eigvalsh(C)
    assert float(eig.min()) > FLOOR[name] / 2
    C.sum().backward()
    assert torch.isfinite(_m[name].raw_tril.grad).all()
print("03 SPD3 modules: PASS (init cov, PSD, CUDA float64, differentiable)")


03 SPD3 modules: PASS (init cov, PSD, CUDA float64, differentiable)


/tmp/ipykernel_34772/4086449628.py:68: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  assert float((C - INIT_STD[name] ** 2 * torch.eye(3, **DD)).abs().max()) < 1e-2 * INIT_STD[name] ** 2


Define covariance regularizers for scale, correlation, condition number, and innovation consistency.


In [4]:
# 04 regularizers: log-eigenvalue prior, correlation, condition number, NIS
LAMBDA = {"prior": 1e-3, "corr": 1e-3, "cond": 1e-1, "nis": 1e-3}
BIAS_PRIOR_BOOST = 1000.0  # hard anchor for Qbg/Qba (see protocol notes above)
MAX_LOG_COND = 6.0

def reg_log_eig_prior(module, prior_std):
    target = 2.0 * torch.log(torch.as_tensor(prior_std, **DD))
    return ((module.log_eigs() - target) ** 2).mean()

def reg_correlation(C):
    d = torch.sqrt(torch.diagonal(C).clamp_min(1e-30))
    Corr = C / (d[:, None] * d[None, :])
    off = Corr - torch.diag(torch.diagonal(Corr))
    return (off ** 2).mean()

def reg_condition_number(C, max_log_cond=MAX_LOG_COND):
    eig = torch.linalg.eigvalsh(C).clamp_min(1e-30)
    log_cond = torch.log(eig[-1]) - torch.log(eig[0])
    return torch.relu(log_cond - max_log_cond) ** 2

def reg_nis(nis_values, nis_dims):
    if len(nis_values) == 0:
        return torch.zeros((), **DD)
    per_dim = torch.stack([n / d for n, d in zip(nis_values, nis_dims)])
    return (per_dim.mean() - 1.0) ** 2

def covariance_regularization(modules, nis_values, nis_dims):
    terms = {}
    total = torch.zeros((), **DD)
    for name in GROUP_ORDER:
        C = modules[name].cov()
        boost = BIAS_PRIOR_BOOST if name in ("gyro_bias", "accel_bias") else 1.0
        r = (LAMBDA["prior"] * boost * reg_log_eig_prior(modules[name], INIT_STD[name])
             + LAMBDA["corr"] * reg_correlation(C)
             + LAMBDA["cond"] * reg_condition_number(C))
        terms[name] = r
        total = total + r
    terms["nis"] = LAMBDA["nis"] * reg_nis(nis_values, nis_dims)
    total = total + terms["nis"]
    return total, terms

# unit checks: at init the prior and correlation terms are ~0; a strongly
# correlated matrix is penalized; an ill-conditioned one trips the cond term
_m = make_cov_modules()
_tot, _terms = covariance_regularization(_m, [], [])
# near zero at init (small residual from the softplus floor offset)
assert float(_tot) < 1e-4, float(_tot)
_corr_mat = torch.tensor([[1.0, 0.9, 0.0], [0.9, 1.0, 0.0], [0.0, 0.0, 1.0]], **DD)
assert abs(float(reg_correlation(_corr_mat)) - 2 * 0.81 / 9) < 1e-12
_ill = torch.diag(torch.tensor([1.0, 1.0, 1e-4], **DD))
assert float(reg_condition_number(_ill)) > 5.0
assert float(reg_nis([torch.tensor(6.0, **DD)], [3])) == 1.0
print("04 regularizers: PASS")


04 regularizers: PASS


Load seven rollouts, trim 1.0 seconds from each end, build fixed contact schedules, and save the initial covariances.


In [5]:
# 05 data: rollouts, trimmed 1.0 s at both ends, precomputed contact events
TRIM_S = 1.0
S_JITTER = 1e-12

def candidate_reliability(p_BC, v_BC, height_scale=0.04, speed_scale=0.35, floor=1e-3):
    z = p_BC[..., 2]
    height_score = np.exp(-(((z - z.min(axis=1, keepdims=True)) / height_scale) ** 2))
    speed_score = np.exp(-((np.linalg.norm(v_BC, axis=-1) / speed_scale) ** 2))
    return np.clip(height_score * speed_score, floor, 1.0)

def hysteresis_contact_schedule(weights, on=0.5, off=0.05):
    T, N = weights.shape
    flags = np.zeros((T, N), dtype=bool)
    state = np.zeros(N, dtype=bool)
    for k in range(T):
        state = np.where(state, weights[k] >= off, weights[k] >= on)
        flags[k] = state
    return flags

def load_rollout(stem):
    d = np.load(DATA_ROOT / f"{stem}.npz")
    f = np.load(DATA_ROOT / f"{stem}.features.npz", allow_pickle=True)
    t = d["meta/sim_time"]
    dt = float(np.median(np.diff(t)))
    p_BC = f["input_kinematics/p_BC"]
    flags = hysteresis_contact_schedule(candidate_reliability(p_BC, f["input_kinematics/v_BC"]))
    # G1 IMU timing (used by this replay convention): row-k IMU drives the
    # interval ending at row k, so imu_step[r] = imu[r].
    imu = np.concatenate([d["input/imu_gyro"], d["input/imu_accelerometer_native"]], axis=1)
    T = p_BC.shape[0]
    n_trim = int(round(TRIM_S / dt))
    trim0, trim1 = n_trim, T - n_trim
    assert trim1 - trim0 > 1000, (stem, trim0, trim1)
    return {
        "stem": stem, "dt": dt, "T": T, "trim0": trim0, "trim1": trim1,
        "imu": torch.as_tensor(imu, **DD),
        "p_BC": torch.as_tensor(p_BC, **DD),
        "flags": flags,
        "changes": precompute_contact_changes(flags),  # per-row contact events
        "gt_R_WB": torch.as_tensor(d["gt/gt_R_WB"], **DD),
        "gt_v_W": torch.as_tensor(d["gt/base_linear_velocity"], **DD),
        "gt_v_B": torch.as_tensor(d["gt/gt_v_B"], **DD),
        "gt_p_W": torch.as_tensor(d["gt/base_position"], **DD),
    }

manifest = json.load(open(DATA_ROOT / "dataset_manifest.json"))
SPLIT_LABEL = {}   # manifest split labels: metadata only
ROLLOUT_ORDER = []
for e in manifest:
    stem = Path(e["dataset_path"]).stem
    if (DATA_ROOT / f"{stem}.npz").exists():
        SPLIT_LABEL[stem] = e["split"]
        ROLLOUT_ORDER.append(stem)
ROLLOUT_ORDER = sorted(ROLLOUT_ORDER)
ROLLOUTS = {stem: load_rollout(stem) for stem in ROLLOUT_ORDER}
assert len(ROLLOUT_ORDER) == 7, ROLLOUT_ORDER

TOTAL_TRAIN_ROWS = sum(r["trim1"] - r["trim0"] - 1 for r in ROLLOUTS.values())
print(f"{len(ROLLOUT_ORDER)} rollouts, trim {TRIM_S}s "
      f"({ROLLOUTS[ROLLOUT_ORDER[0]]['trim0']} rows) each end")
for stem in ROLLOUT_ORDER:
    r = ROLLOUTS[stem]
    print(f"  {stem} [{SPLIT_LABEL[stem]}]: rows {r['trim0']}..{r['trim1']} "
          f"of {r['T']}")
print(f"rows contributing to training loss per epoch: {TOTAL_TRAIN_ROWS} "
      "(the seed row of each rollout is the GT-seeded state itself)")

P0_FIXED = np.eye(15)
for sl, s in [((0, 3), 1e-4), ((3, 6), 1e-2), ((6, 9), 1e-4), ((9, 12), 1e-4), ((12, 15), 1e-2)]:
    P0_FIXED[sl[0]:sl[1], sl[0]:sl[1]] *= s

def seed_state(roll, row):
    """Detached GT seed at the trimmed rollout start (evaluation-only GT use)."""
    X0 = torch.eye(5, **DD)
    X0[0:3, 0:3] = roll["gt_R_WB"][row].detach()
    X0[0:3, 3] = roll["gt_v_W"][row].detach()
    X0[0:3, 4] = roll["gt_p_W"][row].detach()
    return X0.detach(), torch.zeros(6, **DD), torch.as_tensor(P0_FIXED, **DD)

def eval_replay(roll, covs, R_kin, s_jitter=S_JITTER):
    """No-grad continuous replay over the trimmed rollout; returns RMSE,
    sum of squared errors, row count, and final-P health."""
    s0, s1 = roll["trim0"], roll["trim1"]
    with torch.no_grad():
        X0, th0, P0 = seed_state(roll, s0)
        filt = start_filter(X0, th0, P0, covs, roll["flags"][s0],
                            roll["p_BC"][s0], R_kin, s_jitter=s_jitter)
        R0 = filt.X[0:3, 0:3].clone()   # post-correction state at row s0
        v0 = filt.X[0:3, 3].clone()
        out = run_rows(filt, roll["imu"][s0 + 1:s1], roll["dt"],
                       roll["p_BC"][s0 + 1:s1], None, None, R_kin,
                       changes_list=roll["changes"][s0 + 1:s1])
        R_est = torch.cat([R0[None], out["R_WB"]])  # rows s0 .. s1-1
        v_est = torch.cat([v0[None], out["v_W"]])
        v_B = torch.einsum("tji,tj->ti", R_est, v_est)
        se = ((v_B - roll["gt_v_B"][s0:s1]) ** 2).sum(-1)
        rmse = float(torch.sqrt(se.mean()))
        sse = float(se.sum())
        P = filt.P
        sym = float((P - P.T).abs().max())
        min_eig = float(torch.linalg.eigvalsh(0.5 * (P + P.T)).min())
        finite = bool(torch.isfinite(filt.X).all() and torch.isfinite(P).all())
    return {"vB_rmse": rmse, "sse": sse, "rows": int(s1 - s0),
            "final_P_sym": sym, "final_P_min_eig": min_eig,
            "finite": finite, "jitter_events": filt.jitter_events}

# initial (pre-training) covariances: freeze and SAVE before any training
_m0 = make_cov_modules()
with torch.no_grad():
    covs0, Rk0 = build_covs(_m0)
    covs0 = {k: v.detach().clone() for k, v in covs0.items()}
    Rk0 = Rk0.detach().clone()
np.savez(OUT / "initial_covariances.npz",
         **{k: v.cpu().numpy() for k, v in covs0.items()},
         R_kin_pos=Rk0.cpu().numpy())
print("initial covariances saved to", OUT / "initial_covariances.npz")


7 rollouts, trim 1.0s (50 rows) each end
  dance1_subject1_20260623_173019 [train]: rows 50..6524 of 6574
  dance1_subject3_20260623_164158 [val]: rows 50..6524 of 6574
  run1_subject2_20260623_170820 [train]: rows 50..11840 of 11890
  run2_subject1_20260623_165739 [test]: rows 50..12190 of 12240
  walk1_subject5_20260623_164533 [train]: rows 50..13015 of 13065
  walk2_subject3_20260623_171914 [train]: rows 50..11859 of 11909
  walk2_subject4_20260623_163125 [test]: rows 50..11859 of 11909
rows contributing to training loss per epoch: 73454 (the seed row of each rollout is the GT-seeded state itself)
initial covariances saved to runs/covariance_calibration/initial_covariances.npz


Profile one training-shaped chunk to document runtime characteristics without changing the estimator.


In [6]:
# 05b performance profile of one training-shaped chunk (documentation of the
# CPU-bound structure; see the header cell for the full analysis)
_roll = ROLLOUTS[ROLLOUT_ORDER[0]]
_a, _b = _roll["trim0"] + 1, _roll["trim0"] + 201
_L = torch.linalg.cholesky(covs0["Qc"]).requires_grad_(True)
_covs = dict(covs0)

def _bench(with_grad):
    torch.cuda.synchronize(); t0 = time.time()
    _covs["Qc"] = _L @ _L.T
    X0, th0, P0 = seed_state(_roll, _roll["trim0"])
    filt = start_filter(X0, th0, P0, _covs, _roll["flags"][_roll["trim0"]],
                        _roll["p_BC"][_roll["trim0"]], Rk0, s_jitter=S_JITTER)
    ctx = torch.enable_grad() if with_grad else torch.no_grad()
    with ctx:
        out = run_rows(filt, _roll["imu"][_a:_b], _roll["dt"], _roll["p_BC"][_a:_b],
                       None, None, Rk0, changes_list=_roll["changes"][_a:_b])
        if with_grad:
            (out["v_W"] ** 2).sum().backward()
            _L.grad = None
    torch.cuda.synchronize()
    return (time.time() - t0) / (_b - _a) * 1e3

_bench(True)  # warmup
fwd_bwd_ms = _bench(True)
fwd_ms = _bench(False)
print(f"training-shaped chunk: fwd+bwd {fwd_bwd_ms:.1f} ms/step, "
      f"no-grad {fwd_ms:.1f} ms/step")
print("bottleneck: host-side kernel-launch overhead of the dynamic-dimension "
      "sequential filter; backward dominates. Safe caching optimizations are "
      "already applied in src/estimation_calibration_cuda/invariant_ekf.py (values bitwise-verified in "
      "cell 02); further gains need an estimator-level refactor (documented "
      "in the header cell).")


training-shaped chunk: fwd+bwd 7.9 ms/step, no-grad 2.2 ms/step
bottleneck: host-side kernel-launch overhead of the dynamic-dimension sequential filter; backward dominates. Safe caching optimizations are already applied in src/estimation_calibration_cuda/invariant_ekf.py (values bitwise-verified in cell 02); further gains need an estimator-level refactor (documented in the header cell).


Run the trimmed chunked-BPTT training loop and save the selected checkpoint. This cell can be expensive; rerun only when intentionally retraining.


In [7]:
# 06 trimmed training: continuous per-rollout replay, chunked BPTT,
# one Adam update per chunk, selection by epoch-mean training body loss
EPOCHS = 20
CHUNK = 300
BIAS_LR_FACTOR = 1.0 / 30.0   # slow, anchored adaptation for Qbg/Qba

def train(lr):
    torch.manual_seed(0)
    modules = make_cov_modules()
    params = list(modules.parameters())
    bias_params = [modules[n].raw_tril for n in ("gyro_bias", "accel_bias")]
    main_params = [modules[n].raw_tril for n in GROUP_ORDER
                   if n not in ("gyro_bias", "accel_bias")]
    opt = torch.optim.Adam([
        {"params": main_params, "lr": lr},
        {"params": bias_params, "lr": lr * BIAS_LR_FACTOR},
    ], betas=(0.9, 0.999), eps=1e-8)
    history = []
    chunk_trace = []   # per-chunk body loss across all epochs (for convergence plot)
    best = {"loss": float("inf"), "epoch": -1, "state": None}
    t_train = time.time()
    for epoch in range(EPOCHS):
        torch.cuda.reset_peak_memory_stats()
        t_ep = time.time()
        body_losses, reg_losses = [], []
        nis_chunk_means = []                      # GPU tensors, sync at epoch end
        grad_norms = {n: [] for n in GROUP_ORDER}  # GPU tensors, sync at epoch end
        jitter_events = 0
        for stem in ROLLOUT_ORDER:
            roll = ROLLOUTS[stem]
            s0, s_end = roll["trim0"], roll["trim1"]
            covs, R_kin = build_covs(modules)
            X0, th0, P0 = seed_state(roll, s0)
            assert_cuda_float64(X0, P0, R_kin, *covs.values())
            filt = start_filter(X0, th0, P0, covs, roll["flags"][s0],
                                roll["p_BC"][s0], R_kin, s_jitter=S_JITTER)
            a = s0 + 1
            while a < s_end:
                b = min(a + CHUNK, s_end)
                covs, R_kin = build_covs(modules)  # params changed after step
                filt.covs = covs
                out = run_rows(filt, roll["imu"][a:b], roll["dt"],
                               roll["p_BC"][a:b], None, None, R_kin,
                               collect_nis=True, changes_list=roll["changes"][a:b])
                v_B = torch.einsum("tji,tj->ti", out["R_WB"], out["v_W"])
                loss_body = ((v_B - roll["gt_v_B"][a:b]) ** 2).sum(-1).mean()
                loss_reg, _ = covariance_regularization(modules, out["nis_values"],
                                                        out["nis_dims"])
                loss = loss_body + loss_reg
                lb, lg = float(loss_body), float(loss_reg)   # one host sync
                if not (np.isfinite(lb) and np.isfinite(lg)):
                    raise FloatingPointError(f"non-finite loss at epoch {epoch}")
                opt.zero_grad(set_to_none=True)
                loss.backward()
                with torch.no_grad():
                    for n in GROUP_ORDER:
                        g = modules[n].raw_tril.grad
                        grad_norms[n].append(g.norm() if g is not None
                                             else torch.zeros((), **DD))
                    if out["nis_values"]:
                        nis_chunk_means.append(torch.stack(
                            [nv / nd for nv, nd in zip(out["nis_values"],
                                                       out["nis_dims"])]).mean())
                torch.nn.utils.clip_grad_norm_(params, 1.0, error_if_nonfinite=True)
                opt.step()
                body_losses.append(lb)
                reg_losses.append(lg)
                chunk_trace.append(lb)
                detach_filter(filt)
                a = b
            jitter_events += filt.jitter_events   # one sync per rollout
        rec = {
            "epoch": epoch,
            "train_body_loss": float(np.mean(body_losses)),
            "train_reg_loss": float(np.mean(reg_losses)),
            "nis_per_dim_mean": float(torch.stack(nis_chunk_means).mean()),
            "jitter_events": jitter_events,
            "peak_gb": torch.cuda.max_memory_allocated() / 1e9,
            "epoch_s": time.time() - t_ep,
            "groups": {},
        }
        for n in GROUP_ORDER:
            C = modules[n].cov().detach()
            eig = torch.linalg.eigvalsh(C)
            d = torch.sqrt(torch.diagonal(C).clamp_min(1e-30))
            corr = (C / (d[:, None] * d[None, :])).cpu().numpy()
            off = corr - np.diag(np.diag(corr))
            rec["groups"][n] = {
                "eigs": eig.cpu().numpy().tolist(),
                "log_cond": float(torch.log(eig[-1]) - torch.log(eig[0])),
                "max_abs_offdiag_corr": float(np.abs(off).max()),
                "grad_norm_mean": float(torch.stack(grad_norms[n]).mean()),
                "cov": C.cpu().numpy().tolist(),
            }
        history.append(rec)
        if rec["train_body_loss"] < best["loss"]:
            best = {"loss": rec["train_body_loss"], "epoch": epoch,
                    "state": copy.deepcopy(modules.state_dict())}
        print(f"epoch {epoch:2d}: body {rec['train_body_loss']:.4f} "
              f"reg {rec['train_reg_loss']:.5f} | NIS/dim {rec['nis_per_dim_mean']:.3f} "
              f"| {rec['epoch_s']:.0f}s ({TOTAL_TRAIN_ROWS / rec['epoch_s']:.0f} rows/s) "
              f"| peak {rec['peak_gb']:.2f} GB")
    return modules, opt, history, best, chunk_trace, time.time() - t_train

# Use the completed run's learning rate; retry lower only on non-finite loss.
LR = 1e-2
try:
    cov_modules, optimizer, history, best, chunk_trace, train_runtime_s = train(LR)
except FloatingPointError as e:
    print(f"non-finite loss at lr={LR} ({e}); retrying with lr=3e-3")
    LR = 3e-3
    cov_modules, optimizer, history, best, chunk_trace, train_runtime_s = train(LR)

final_state = copy.deepcopy(cov_modules.state_dict())   # last-epoch state
cov_modules.load_state_dict(best["state"])              # selected checkpoint
RUN_META = {
        "trim_s": TRIM_S,
    "train_rollouts": ROLLOUT_ORDER,
    "manifest_split_labels": SPLIT_LABEL,
    "total_trained_rows_per_epoch": TOTAL_TRAIN_ROWS,
    "chunk_size": CHUNK,
    "epochs": EPOCHS,
    "lr": LR,
    "bias_lr_factor": BIAS_LR_FACTOR,
    "selected_epoch": best["epoch"],
    "selected_train_body_loss": best["loss"],
    "train_runtime_s": train_runtime_s,
    "cuda_peak_gb": max(h["peak_gb"] for h in history),
    "device": torch.cuda.get_device_name(0),
}
torch.save({
    **RUN_META,
    "selected_state_dict": best["state"],
    "final_state_dict": final_state,
    "optimizer_state_dict": optimizer.state_dict(),
    "init_std": INIT_STD, "floor": FLOOR, "lambda": LAMBDA,
    "bias_prior_boost": BIAS_PRIOR_BOOST,
}, OUT / "calibration_checkpoint.pt")
print(f"selected epoch {best['epoch']} by train body loss {best['loss']:.4f} "
      f"(epoch 0: {history[0]['train_body_loss']:.4f}) | "
      f"runtime {train_runtime_s / 60:.1f} min")
print("checkpoint (selected + final + optimizer state) saved to",
      OUT / "calibration_checkpoint.pt")
assert best["loss"] < history[0]["train_body_loss"], "training loss must decrease"
assert all(np.isfinite([h["train_body_loss"] for h in history]))
for n in GROUP_ORDER:
    gmean = np.mean([h["groups"][n]["grad_norm_mean"] for h in history])
    print(f"mean grad norm [{n}]: {gmean:.3e}")
assert np.mean([h["groups"]["kin_meas"]["grad_norm_mean"] for h in history]) > 0
assert np.mean([h["groups"]["contact_proc"]["grad_norm_mean"] for h in history]) > 0


epoch  0: body 3.7250 reg 0.00183 | NIS/dim 0.126 | 246s (298 rows/s) | peak 0.16 GB


epoch  1: body 3.2996 reg 0.00374 | NIS/dim 0.108 | 246s (298 rows/s) | peak 0.16 GB


epoch  2: body 2.9987 reg 0.00572 | NIS/dim 0.093 | 247s (297 rows/s) | peak 0.16 GB


epoch  3: body 2.8811 reg 0.00723 | NIS/dim 0.085 | 247s (298 rows/s) | peak 0.16 GB


epoch  4: body 2.8450 reg 0.00826 | NIS/dim 0.081 | 247s (297 rows/s) | peak 0.16 GB


...

epoch 15: body 4.9874 reg 0.01462 | NIS/dim 0.058 | 247s (297 rows/s) | peak 0.16 GB


epoch 16: body 5.1060 reg 0.01490 | NIS/dim 0.055 | 246s (298 rows/s) | peak 0.16 GB


epoch 17: body 5.2051 reg 0.01515 | NIS/dim 0.054 | 246s (299 rows/s) | peak 0.16 GB


epoch 18: body 5.2903 reg 0.01538 | NIS/dim 0.052 | 244s (301 rows/s) | peak 0.16 GB


epoch 19: body 5.3648 reg 0.01559 | NIS/dim 0.051 | 245s (300 rows/s) | peak 0.16 GB
selected epoch 5 by train body loss 2.8429 (epoch 0: 3.7250) | runtime 82.1 min
checkpoint (selected + final + optimizer state) saved to runs/covariance_calibration/calibration_checkpoint.pt
mean grad norm [gyro]: 3.221e-02
mean grad norm [accel]: 8.876e-02
mean grad norm [gyro_bias]: 2.153e-05
mean grad norm [accel_bias]: 2.238e-05
mean grad norm [contact_proc]: 1.540e-01
mean grad norm [kin_meas]: 2.018e-01
